In [1]:
import qibo
from qibo import Circuit, gates, set_backend
import time
set_backend("qibojit", platform="cuda")

[Qibo 0.3.5|INFO|2026-11-04 15:29:04]: Using qibojit (numba) backend on /CPU:0


In [2]:
def initialize_sqrt(_circ, x, n, allocs):
    R, F = allocs
    if not 0 <= x < 2**n:
        raise ValueError(f"x={x} does not fit in {n} bits")
    for i in range(n):
        if (x >> i) & 1:
            _circ.add(gates.X(R[i]))

In [3]:
from square_opt import *
n = 10
num_qubits = n + 2
h = n//2

start = time.time()
for x in range(2**n):
    R, F = get_allocs(n)
    #print('==========================================')
    #print(f'R: {R}, F:{F}')

    sqrt, (R, F) = sqrt_circ(n, (R,F))
    #print(f'R: {R}, F:{F}')
    circ = qibo.Circuit(num_qubits)
    initialize_sqrt(circ, x, n, (R,F))
    
    circ.add(sqrt.on_qubits(*range(num_qubits)))
    circ.add(gates.M(*range(num_qubits)))

    res = circ(nshots=1)

    samples = res.samples()[0]
    sqrt_bits = [samples[F[k]] for k in range(2, h+2)]
    remainder = [samples[R[k]] for k in range(0, h+1)]
    sqrt_value = sum(bit << i for i, bit in enumerate(sqrt_bits))
    remainder_value = sum(bit << i for i, bit in enumerate(remainder))
    print(f'Input: {x} | Result: {sqrt_value} | Remainder: {remainder_value} | Output: {samples}')
end = time.time()
print(f'Elapsed time for {2**n} entries: {end-start:0.4f}')

Input: 0 | Result: 0 | Remainder: 0 | Output: [0 0 0 0 0 0 0 0 0 0 0 0]
Input: 1 | Result: 1 | Remainder: 0 | Output: [0 0 0 0 0 0 1 0 0 0 0 0]
Input: 2 | Result: 1 | Remainder: 1 | Output: [1 0 0 0 0 0 1 0 0 0 0 0]
Input: 3 | Result: 1 | Remainder: 2 | Output: [0 1 0 0 0 0 1 0 0 0 0 0]
Input: 4 | Result: 2 | Remainder: 0 | Output: [0 0 0 0 0 0 0 1 0 0 0 0]
Input: 5 | Result: 2 | Remainder: 1 | Output: [1 0 0 0 0 0 0 1 0 0 0 0]
Input: 6 | Result: 2 | Remainder: 2 | Output: [0 1 0 0 0 0 0 1 0 0 0 0]
Input: 7 | Result: 2 | Remainder: 3 | Output: [1 1 0 0 0 0 0 1 0 0 0 0]
Input: 8 | Result: 2 | Remainder: 4 | Output: [0 0 1 0 0 0 0 1 0 0 0 0]
Input: 9 | Result: 3 | Remainder: 0 | Output: [0 0 0 0 0 0 1 1 0 0 0 0]
Input: 10 | Result: 3 | Remainder: 1 | Output: [1 0 0 0 0 0 1 1 0 0 0 0]
Input: 11 | Result: 3 | Remainder: 2 | Output: [0 1 0 0 0 0 1 1 0 0 0 0]
Input: 12 | Result: 3 | Remainder: 3 | Output: [1 1 0 0 0 0 1 1 0 0 0 0]
Input: 13 | Result: 3 | Remainder: 4 | Output: [0 0 1 0 0 0 1